## Dependencies

In [2]:
import numpy as np
import pandas as pd

from bioservices import KEGG
import re
import random
import GEOparse
import mygene
from mygene import MyGeneInfo
from collections import defaultdict, Counter

Creating directory /root/.config/bioservices 


## GSE48558 Dataset from GEO

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE48558

In [3]:
# Download and parse the dataset
gse = GEOparse.get_GEO("GSE48558", destdir=".", silent=True)

In [ ]:
print(gse)

<SERIES: GSE48558 - 170 SAMPLES, 1 d(s)>


In [4]:
# List all sample IDs
print(gse.gsms.keys())

# Inspect the first sample’s metadata
first_sample = list(gse.gsms.values())[0]
print(first_sample.metadata)

dict_keys(['GSM1180750', 'GSM1180751', 'GSM1180752', 'GSM1180753', 'GSM1180754', 'GSM1180755', 'GSM1180756', 'GSM1180757', 'GSM1180758', 'GSM1180759', 'GSM1180760', 'GSM1180761', 'GSM1180762', 'GSM1180763', 'GSM1180764', 'GSM1180765', 'GSM1180766', 'GSM1180767', 'GSM1180768', 'GSM1180769', 'GSM1180770', 'GSM1180771', 'GSM1180772', 'GSM1180773', 'GSM1180774', 'GSM1180775', 'GSM1180776', 'GSM1180777', 'GSM1180778', 'GSM1180779', 'GSM1180780', 'GSM1180781', 'GSM1180782', 'GSM1180783', 'GSM1180784', 'GSM1180785', 'GSM1180786', 'GSM1180787', 'GSM1180788', 'GSM1180789', 'GSM1180790', 'GSM1180791', 'GSM1180792', 'GSM1180793', 'GSM1180794', 'GSM1180795', 'GSM1180796', 'GSM1180797', 'GSM1180798', 'GSM1180799', 'GSM1180800', 'GSM1180801', 'GSM1180802', 'GSM1180803', 'GSM1180804', 'GSM1180805', 'GSM1180806', 'GSM1180807', 'GSM1180808', 'GSM1180809', 'GSM1180810', 'GSM1180811', 'GSM1180812', 'GSM1180813', 'GSM1180814', 'GSM1180815', 'GSM1180816', 'GSM1180817', 'GSM1180818', 'GSM1180819', 'GSM11808

In [5]:
# Build metadata list
meta_rows = []
for gsm_name, gsm in gse.gsms.items():
    # get source_name_ch1 if exists, otherwise fallback to title or 'Unknown'
    src = gsm.metadata.get("source_name_ch1", None)
    if src:
        source_name = src[0]
    else:
        title = gsm.metadata.get("title", [""])[0]
        source_name = title or "Unknown"

    meta_rows.append({
        "sample_id": gsm_name,
        "source_name": source_name
    })

meta_df = pd.DataFrame(meta_rows).set_index("sample_id")

In [6]:
print(meta_df["source_name"].value_counts())

source_name
B ALL Cell Line    27
B ALL Patient      27
AML Cell Line      21
AML Patient        18
T Cells            17
T ALL Cell Line    15
T ALL Patient      13
Granulocytes       12
B Cells            11
Monocytes           6
CD34+HSPC           3
Name: count, dtype: int64


In [7]:
# build expression dataframe (samples x probes)
expr_df = pd.DataFrame({
    gsm_name: gsm.table.set_index("ID_REF")["VALUE"].astype(float)
    for gsm_name, gsm in gse.gsms.items()
}).transpose()

print("Total samples in expr_df:", expr_df.shape[0], "probes:", expr_df.shape[1])

Total samples in expr_df: 170 probes: 32321


In [8]:
# filter only "B ALL Patient" and "AML Patient"
keep_classes = ["b all patient", "aml patient"]
meta_df["source_name_clean"] = meta_df["source_name"].astype(str).str.strip().str.lower()

mask = meta_df["source_name_clean"].isin(keep_classes)
meta_keep = meta_df[mask].copy()

print("Samples matching 'B ALL Patient' or 'AML Patient':", meta_keep.shape[0])
print(meta_keep["source_name"].value_counts())

Samples matching 'B ALL Patient' or 'AML Patient': 45
source_name
B ALL Patient    27
AML Patient      18
Name: count, dtype: int64


In [9]:
selected_samples = [s for s in meta_keep.index if s in expr_df.index]
print("Selected samples present in expression data:", len(selected_samples))

Selected samples present in expression data: 45


In [10]:
X_df = expr_df.loc[selected_samples].copy()

In [11]:
# Change label AML -> 1, ALL -> 0
def label_from_source(clean_name):
    if "aml" in clean_name:
        return 1
    if "all" in clean_name:
        return 0

labels = meta_keep.loc[selected_samples, "source_name_clean"].apply(label_from_source)

print("Label distribution (0=ALL, 1=AML):")
print(labels.value_counts(dropna=False))

Label distribution (0=ALL, 1=AML):
source_name_clean
0    27
1    18
Name: count, dtype: int64


In [12]:
# Add labels as the last column of new df
X_df_labeled = X_df.assign(label=labels.values)
X_df_labeled["label"] = X_df_labeled["label"].astype(int)

print("Final dataframe shape (samples x probes+1):", X_df_labeled.shape)
print(X_df_labeled.head(3))

Final dataframe shape (samples x probes+1): (45, 32322)
ID_REF       7892501   7892502   7892503   7892504   7892505   7892506  \
GSM1180750  5.635547  6.640414  5.108161  8.414047  2.280691  4.123770   
GSM1180751  4.916813  5.838517  5.953453  9.026401  2.423883  5.783080   
GSM1180752  5.478152  7.101921  6.383739  9.456269  3.141614  4.898264   

ID_REF       7892507   7892508    7892509   7892510  ...    8180409  \
GSM1180750  3.894260  6.138993  10.743463  3.692390  ...  11.496730   
GSM1180751  3.538837  6.827082  10.374949  4.667512  ...  11.010435   
GSM1180752  3.589695  6.837599  11.340184  6.292694  ...  11.408984   

ID_REF        8180410    8180411   8180413   8180414   8180415   8180416  \
GSM1180750   9.971627   9.299441  7.439705  4.649011  6.537406  6.165602   
GSM1180751  10.028696   9.302182  6.780970  4.821685  6.010390  4.324456   
GSM1180752  10.201044  10.012556  5.729117  4.451156  6.680938  6.436732   

ID_REF       8180417   8180418  label  
GSM1180750  7.594

In [13]:
X_df_labeled.to_csv("original.csv", index=True)
print(f"Saved to csv.")

Saved to csv.


## Map probe -> entrez -> gene symbol

In [14]:
# Fetch the GPL6244 platform
gpl = GEOparse.get_GEO("GPL6244", destdir=".", silent=True)
gpl_table = gpl.table
print("GPL6244 table shape:", gpl_table.shape)
print("Columns:", list(gpl_table.columns)[:15])

GPL6244 table shape: (33297, 12)
Columns: ['ID', 'GB_LIST', 'SPOT_ID', 'seqname', 'RANGE_GB', 'RANGE_STRAND', 'RANGE_START', 'RANGE_STOP', 'total_probes', 'gene_assignment', 'mrna_assignment', 'category']


In [ ]:
mg = mygene.MyGeneInfo()

# extract Entrez IDs from GPL gene_assignment
def extract_entrez_from_assignment(s):
    if pd.isna(s):
        return None
    parts = [p.strip() for p in str(s).split("//")]

    if len(parts) >= 2 and re.fullmatch(r"\d+", parts[1]):
        return parts[1]
    # fallback
    for p in parts[:3]:
        if re.fullmatch(r"\d+", p):
            return p
    m = re.search(r"\b(\d{1,10})\b", s)
    return m.group(1) if m else None

probe_to_entrez = {}
for _, row in gpl_table.iterrows():
    pid = str(row['ID']).strip()
    ent = extract_entrez_from_assignment(row.get('gene_assignment', None))
    if ent:
        probe_to_entrez[pid] = ent

entrez_ids = sorted({v for v in probe_to_entrez.values() if v and str(v).isdigit()})

def query_entrez_to_symbol(entrez_list, batch=1000):
    out = {}
    for i in range(0, len(entrez_list), batch):
        chunk = entrez_list[i:i+batch]
        res = mg.getgenes(chunk, fields='symbol', as_dataframe=False)
        for r in res:
            q = str(r.get('entrezgene') or r.get('query'))
            sym = r.get('symbol')
            out[q] = sym.upper() if sym else None
    return out

def parse_gene_assignment_symbol(s):
    if pd.isna(s):
        return None
    parts = [p.strip() for p in str(s).split("//")]
    for p in parts[:3]:
        if re.fullmatch(r"[A-Za-z][A-Za-z0-9\-_]*", p) and not re.match(r'^(ENST|ENSG|NM_|NR_|XM_|XR_|ENS|AB)\d+', p, re.I):
            return p.upper()
    return None

In [16]:
entrez2sym = query_entrez_to_symbol(entrez_ids)

# Build probe -> symbol mapping
probe_to_symbol = {}
for _, row in gpl_table.iterrows():
    pid = str(row['ID']).strip()
    ga = row.get('gene_assignment', None)
    sym_from_assign = parse_gene_assignment_symbol(ga)
    if sym_from_assign:
        probe_to_symbol[pid] = sym_from_assign
        continue
    ent = probe_to_entrez.get(pid) if 'probe_to_entrez' in globals() else extract_entrez_from_assignment(ga)
    probe_to_symbol[pid] = entrez2sym.get(ent) if (ent and str(ent).isdigit()) else None

INFO:biothings.client:querying 1-1000 ...
INFO:biothings.client:querying 1-1000 ...
INFO:biothings.client:querying 1-718 ...


In [17]:
# Apply mapping subset_df
df = X_df_labeled.copy()

# remove label column
labels = df["label"].copy()
df_no_label = df.drop(columns=["label"])

# mapping keys to strings
df_no_label.columns = [str(c).strip() for c in df_no_label.columns]
probe_to_symbol_norm = {str(k).strip(): (v.upper().strip() if v else None) for k, v in probe_to_symbol.items()}

# identify probes present in expression matrix that have a mapped symbol
probes_in_expr = [p for p in df_no_label.columns if p in probe_to_symbol_norm and probe_to_symbol_norm[p]]
print(f"Probes in expression data: {len(df_no_label.columns)}")
print(f"Probes with mapping in platform annotation: {len([k for k in probe_to_symbol_norm.keys()])}")
print(f"Probes present in expression data AND mapped to symbol: {len(probes_in_expr)}")

Probes in expression data: 32321
Probes with mapping in platform annotation: 33297
Probes present in expression data AND mapped to symbol: 24520


In [18]:
# subset & rename columns to gene symbols
rename_map = {p: probe_to_symbol_norm[p] for p in probes_in_expr}
mapped_df = df_no_label[probes_in_expr].rename(columns=rename_map)

# collapse probes that map to the same symbol by mean
gene_symbol_df = mapped_df.groupby(level=0, axis=1).mean()

# reattach label column
gene_symbol_df = gene_symbol_df.copy()
gene_symbol_df["label"] = labels.loc[gene_symbol_df.index].values

/tmp/ipython-input-1521062465.py:6: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  gene_symbol_df = mapped_df.groupby(level=0, axis=1).mean()


In [19]:
gene_symbol_df.head()

,A1BG,A1CF,A2M,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,label
GSM1180750,5.514638,3.586495,4.153631,3.449145,4.431828,3.792182,5.154059,3.357011,8.419307,5.739747,...,7.210440,8.760705,6.759785,7.678924,3.390258,6.739212,9.405321,8.037282,7.408837,1
GSM1180751,5.239642,3.645632,3.330942,4.266428,4.698815,3.951576,4.891572,3.881203,8.164883,6.049630,...,5.605671,8.594286,7.547800,7.915605,3.569541,7.552835,10.133702,7.820980,7.092259,1
GSM1180752,5.540831,3.696779,4.308437,3.950829,4.623143,4.213858,4.865458,3.280896,7.934749,5.936680,...,6.507919,8.457704,7.346999,8.112584,3.280896,7.121801,9.624205,7.752162,7.911634,1
GSM1180753,5.421676,3.962463,3.654300,3.962463,4.701298,4.435051,5.606486,3.721220,8.240141,5.443177,...,6.217405,9.170125,7.336326,8.143446,3.654300,7.024605,10.357758,7.800852,6.661170,1
GSM1180754,5.237251,3.523454,3.966461,4.261344,4.046966,4.092443,4.704431,3.966461,8.662806,6.363571,...,7.031296,7.953477,6.943295,7.760145,3.477393,7.512131,9.208405,7.736115,7.726616,1


In [20]:
gene_symbol_df.shape

(45, 23381)

In [21]:
gene_symbol_df.to_csv("original_mapped.csv", index=True)
print("saved csv")

saved csv


In [22]:
gene_symbols = {c for c in gene_symbol_df.columns if c != "label"}

## Get genes from KEGG cancer pathways

In [ ]:
unique_intersect_gene = set()

In [ ]:
def fetch_kegg_aml_symbols(pathway):
    k = KEGG()
    entry = k.get(pathway)
    parsed = k.parse(entry)
    symbols = set()
    if 'GENE' in parsed:
        for entrez, desc in parsed['GENE'].items():
            m = re.match(r"^([A-Za-z0-9\-\._]+)", desc)
            if m:
                symbols.add(m.group(1).upper().strip())
    return symbols

def add_to_set(list):
    for item in list:
      if item not in unique_intersect_gene:
        unique_intersect_gene.add(item)
      pass

def get_intersect_genes(gene_symbols, pathway):
    symbols_in_pathway = fetch_kegg_aml_symbols(pathway)
    print(f'Number of symbols in {pathway}: {len(symbols_in_pathway)}')

    intersected_genes = sorted(list(gene_symbols.intersection(symbols_in_pathway)))
    print("Matched genes:", len(intersected_genes))

    add_to_set(intersected_genes)
    print("Number of intersected gene: ", len(unique_intersect_gene))

In [ ]:
kegg_pathway_list = [
    'hsa05200',
    'hsa05205',
    'hsa05206',
    'hsa05202',
    'hsa05223',
    'hsa05230',
    'hsa05222',
    'hsa05224',
    'hsa05215',
    'hsa05213',
    'hsa05226',
    'hsa05210',
    'hsa05212',
    'hsa05216',
    'hsa05219',
    'hsa05231',
    'hsa05235',
    'hsa05217',
    'hsa05208',
    'hsa05203',
    'hsa05169',
    'hsa05211',
    'hsa05204',
    'hsa05225',
    'hsa05207',
    'hsa05218',
    'hsa05214',
    'hsa05165',
    'hsa04371',
    'hsa03460',
    'hsa03320'
]

In [ ]:
for i in kegg_pathway_list:
  get_intersect_genes(gene_symbols, i)

Number of symbols in hsa05200: 533
Matched genes: 520
Number of intersected gene:  520
Number of symbols in hsa05205: 204
Matched genes: 199
Number of intersected gene:  610
Number of symbols in hsa05206: 320
Matched genes: 278
Number of intersected gene:  795
Number of symbols in hsa05202: 201
Matched genes: 174
Number of intersected gene:  905
Number of symbols in hsa05223: 73
Matched genes: 72
Number of intersected gene:  910
Number of symbols in hsa05230: 71
Matched genes: 69
Number of intersected gene:  942
Number of symbols in hsa05222: 93
Matched genes: 92
Number of intersected gene:  942
Number of symbols in hsa05224: 148
Matched genes: 147
Number of intersected gene:  948
Number of symbols in hsa05215: 106
Matched genes: 105
Number of intersected gene:  969
Number of symbols in hsa05213: 59
Matched genes: 58
Number of intersected gene:  970
Number of symbols in hsa05226: 150
Matched genes: 149
Number of intersected gene:  974
Number of symbols in hsa05210: 87
Matched genes: 85

In [ ]:
with open("kegg_intersect.txt", "w") as f:
  f.write(repr(unique_intersect_gene))

## Extract subset from intersected genes

In [23]:
df = pd.read_csv('original_mapped.csv')
df.head(5)

,Unnamed: 0,A1BG,A1CF,A2M,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,AAAS,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,label
0,GSM1180750,5.514638,3.586495,4.153631,3.449145,4.431828,3.792182,5.154059,3.357011,8.419307,...,7.210440,8.760705,6.759785,7.678924,3.390258,6.739212,9.405321,8.037282,7.408837,1
1,GSM1180751,5.239642,3.645632,3.330942,4.266428,4.698815,3.951576,4.891572,3.881203,8.164883,...,5.605671,8.594286,7.547800,7.915605,3.569541,7.552835,10.133702,7.820980,7.092259,1
2,GSM1180752,5.540831,3.696779,4.308437,3.950829,4.623143,4.213858,4.865458,3.280896,7.934749,...,6.507919,8.457704,7.346999,8.112584,3.280896,7.121801,9.624205,7.752162,7.911634,1
3,GSM1180753,5.421676,3.962463,3.654300,3.962463,4.701298,4.435051,5.606486,3.721220,8.240141,...,6.217405,9.170125,7.336326,8.143446,3.654300,7.024605,10.357758,7.800852,6.661170,1
4,GSM1180754,5.237251,3.523454,3.966461,4.261344,4.046966,4.092443,4.704431,3.966461,8.662806,...,7.031296,7.953477,6.943295,7.760145,3.477393,7.512131,9.208405,7.736115,7.726616,1


In [24]:
with open("kegg_intersect.txt") as f:
    initial_intersect = eval(f.read())

print(len(initial_intersect))
print(list(initial_intersect)[:10])

1701
['PKM', 'CYP2C19', 'PRKAB1', 'GSTA4', 'EPO', 'GNAQ', 'MIRLET7A1', 'DDX3X', 'CSF3R', 'PDGFA']


In [25]:
df.columns = [str(c).strip() for c in df.columns]
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [26]:
df.head()

,A1BG,A1CF,A2M,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,label
0,5.514638,3.586495,4.153631,3.449145,4.431828,3.792182,5.154059,3.357011,8.419307,5.739747,...,7.210440,8.760705,6.759785,7.678924,3.390258,6.739212,9.405321,8.037282,7.408837,1
1,5.239642,3.645632,3.330942,4.266428,4.698815,3.951576,4.891572,3.881203,8.164883,6.049630,...,5.605671,8.594286,7.547800,7.915605,3.569541,7.552835,10.133702,7.820980,7.092259,1
2,5.540831,3.696779,4.308437,3.950829,4.623143,4.213858,4.865458,3.280896,7.934749,5.936680,...,6.507919,8.457704,7.346999,8.112584,3.280896,7.121801,9.624205,7.752162,7.911634,1
3,5.421676,3.962463,3.654300,3.962463,4.701298,4.435051,5.606486,3.721220,8.240141,5.443177,...,6.217405,9.170125,7.336326,8.143446,3.654300,7.024605,10.357758,7.800852,6.661170,1
4,5.237251,3.523454,3.966461,4.261344,4.046966,4.092443,4.704431,3.966461,8.662806,6.363571,...,7.031296,7.953477,6.943295,7.760145,3.477393,7.512131,9.208405,7.736115,7.726616,1


In [27]:
# keep only intersected genes that are present in the df
genes_present = [g for g in initial_intersect if g in df.columns]
missing_genes = [g for g in initial_intersect if g not in df.columns]
cols_to_keep = genes_present + ["label"]

print(f"Sampled genes found in dataframe: {len(genes_present)}")

Sampled genes found in dataframe: 1701


In [28]:
reduced_df = df.loc[:, cols_to_keep].copy()

print("Reduced dataframe shape (rows x cols):", reduced_df.shape)
print("Label distribution in reduced set:")
print(reduced_df["label"].value_counts())

Reduced dataframe shape (rows x cols): (45, 1702)
Label distribution in reduced set:
label
0    27
1    18
Name: count, dtype: int64


In [29]:
reduced_df.to_csv("reduced.csv", index=False)
print(f"Saved reduced dataframe")

Saved reduced dataframe
